In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
import cv2, os

video_path = "/content/drive/MyDrive/traffic_project/videos/no_accident_1.mp4"

save_dir = "/content/drive/MyDrive/traffic_project/Accident_dataset/data/train/Non Accident"
os.makedirs(save_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
saved = 0

for i in range(0, total, 15):   # save every 15th frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ret, frame = cap.read()

    if ret:
        path = os.path.join(save_dir, f"hardneg_truck_{i:04d}.jpg")
        cv2.imwrite(path, frame)
        saved += 1

cap.release()
print(f"✅ Saved {saved} hard negative frames")

In [ ]:

# EfficientNet-B0 + LSTM Accident Detection

!pip install -q timm gradio opencv-python-headless

import os
import cv2
import json
import math
import random
import tempfile
import numpy as np
from PIL import Image
from glob import glob

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
import timm

import matplotlib.pyplot as plt
import gradio as gr
from collections import deque


# CONFIG

BASE = "/content/drive/MyDrive/traffic_project"
DATA_PATH = f"{BASE}/Accident_dataset/data"

SEQ_LEN = 10
IMG_SIZE = 224
BATCH_SIZE = 4
NUM_CLASSES = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


# TRANSFORMS

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])


# BUILD SEQUENCES FROM IMAGE FOLDERS

class SequenceFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None, seq_len=10):
        self.transform = transform
        self.seq_len = seq_len
        self.samples = []
        self.class_names = sorted(os.listdir(root_dir))

        for label_idx, cls in enumerate(self.class_names):
            cls_path = os.path.join(root_dir, cls)
            imgs = sorted(glob(os.path.join(cls_path, "*")))

            # make chunks of seq_len
            for i in range(0, len(imgs) - seq_len + 1, seq_len):
                seq = imgs[i:i+seq_len]
                self.samples.append((seq, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq_paths, label = self.samples[idx]
        frames = []

        for p in seq_paths:
            img = Image.open(p).convert("RGB")
            img = self.transform(img)
            frames.append(img)

        frames = torch.stack(frames)   # [T,C,H,W]
        return frames, label


# DATASETS

train_ds = SequenceFolderDataset(
    os.path.join(DATA_PATH, "train"),
    transform=train_tf,
    seq_len=SEQ_LEN
)

val_ds = SequenceFolderDataset(
    os.path.join(DATA_PATH, "val"),
    transform=val_tf,
    seq_len=SEQ_LEN
)

test_ds = SequenceFolderDataset(
    os.path.join(DATA_PATH, "test"),
    transform=val_tf,
    seq_len=SEQ_LEN
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_ds.class_names
print("Classes:", class_names)
print("Train sequences:", len(train_ds))
print("Val sequences:", len(val_ds))
print("Test sequences:", len(test_ds))

In [ ]:

#  MODEL: EfficientNet-B0 + LSTM


import torch
import torch.nn as nn
import timm

class EfficientNetLSTM(nn.Module):
    def __init__(self, num_classes=2, hidden_size=128, num_layers=1, dropout=0.3):
        super(EfficientNetLSTM, self).__init__()

        # EfficientNet feature extractor
        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0   # remove classifier head
        )

        self.feature_dim = self.backbone.num_features

        # LSTM
        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0 if num_layers == 1 else dropout
        )

        # Final classifier
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        """
        x shape: [B, T, C, H, W]
        B = batch size
        T = sequence length
        """

        B, T, C, H, W = x.shape

        # reshape for CNN
        x = x.view(B * T, C, H, W)

        # EfficientNet features
        feats = self.backbone(x)   # [B*T, F]

        # reshape back to sequence
        feats = feats.view(B, T, -1)   # [B, T, F]

        # LSTM output
        out, _ = self.lstm(feats)

        # last timestep output
        out = out[:, -1, :]

        out = self.dropout(out)
        out = self.fc(out)

        return out



# CREATE MODEL

model = EfficientNetLSTM(
    num_classes=2,
    hidden_size=128,
    num_layers=1,
    dropout=0.3
).to(device)

print(model)
print("✅ EfficientNet + LSTM model created")

In [ ]:

# — TRAINING + VALIDATION + SAVE BEST MODEL (FIXED)


import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

EPOCHS = 10
best_acc = 0.0
SAVE_PATH = f"{BASE}/efficientnet_lstm_best.pth"

for epoch in range(EPOCHS):

    # ---------------- TRAIN ----------------
    model.train()
    train_correct = 0
    train_total = 0
    train_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for sequences, labels in pbar:
        sequences = sequences.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(sequences)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (preds == labels).sum().item()

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100*train_correct/train_total:.2f}%"
        })

    train_acc = 100 * train_correct / train_total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for sequences, labels in val_loader:
            sequences = sequences.to(device)
            labels = labels.to(device)

            outputs = model(sequences)
            _, preds = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    scheduler.step(val_acc)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Acc: {train_acc:.2f}%")
    print(f"Val Acc  : {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print("✅ Best model saved")

print("\n🎉 Training Completed")
print(f"🏆 Best Validation Accuracy: {best_acc:.2f}%")

In [ ]:

# Smart Traffic Accident Detection
# EfficientNet + LSTM + Premium UI + Alarm


!pip -q install gradio opencv-python-headless matplotlib timm

import os, cv2, json, torch, tempfile, numpy as np, gradio as gr
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import deque
from torchvision import transforms
import timm
from PIL import Image


# CONFIG

MODEL_PATH = "/content/drive/MyDrive/traffic_project/efficientnet_lstm_best.pth"
MAP_PATH   = "/content/drive/MyDrive/traffic_project/class_mapping.json"

IMG_SIZE = 224
SEQ_LEN  = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", device)


# LOAD CLASS MAPPING

with open(MAP_PATH, "r") as f:
    mapping = json.load(f)

class_names = mapping["class_names"]
ACCIDENT_IDX = mapping["class_to_idx"]["Accident"]


# TRANSFORM

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


# MODEL

class EfficientNetLSTM(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=False,
            num_classes=0
        )

        self.feature_dim = self.backbone.num_features

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=128,   # ✅ match trained model
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(128, num_classes)   # ✅ match trained model

    def forward(self, x):
        B,T,C,H,W = x.shape
        x = x.view(B*T,C,H,W)

        feat = self.backbone(x)
        feat = feat.view(B,T,-1)

        out,_ = self.lstm(feat)
        out = out[:,-1,:]

        out = self.dropout(out)
        out = self.fc(out)

        return out

model = EfficientNetLSTM(num_classes=2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

print("✅ Model Loaded")

# CSS UI

CSS = """
body{
    background: linear-gradient(135deg,#050816,#0f172a,#111827);
}
.gradio-container{
    font-family: 'Inter', sans-serif;
    color:white;
}
.main-title{
    text-align:center;
    font-size:38px;
    font-weight:800;
    background:linear-gradient(90deg,#ff416c,#ff4b2b,#7f5af0);
    -webkit-background-clip:text;
    -webkit-text-fill-color:transparent;
    margin-bottom:8px;
}
.sub-title{
    text-align:center;
    color:#cbd5e1;
    font-size:16px;
    margin-bottom:20px;
}
.card{
    background:rgba(255,255,255,0.06);
    border:1px solid rgba(255,255,255,0.08);
    backdrop-filter: blur(12px);
    border-radius:18px;
    padding:14px;
}
button{
    border-radius:14px !important;
    font-weight:700 !important;
}
"""


# GRAPH

def make_graph(vals, threshold):
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(vals, linewidth=2)
    ax.axhline(threshold, linestyle="--")
    ax.set_title("Accident Confidence Over Time")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Probability")
    plt.tight_layout()
    return fig


# ALARM HTML

def alarm_html(trigger=False):
    if not trigger:
        return "<div style='padding:15px;color:gray;'>No Alarm</div>"

    return """
    <div style="
        background:linear-gradient(90deg,#ff0000,#ff5e5e);
        color:white;
        padding:20px;
        border-radius:15px;
        text-align:center;
        font-size:28px;
        font-weight:900;
        animation: blink 1s infinite;
    ">
    🚨 ACCIDENT DETECTED 🚨
    <audio autoplay>
        <source src="https://actions.google.com/sounds/v1/alarms/alarm_clock.ogg" type="audio/ogg">
    </audio>
    </div>

    <style>
    @keyframes blink{
        0%{opacity:1;}
        50%{opacity:0.4;}
        100%{opacity:1;}
    }
    </style>
    """


# DETECT FUNCTION

def detect(video, threshold):
    if video is None:
        return None, "Upload Video", None, "No Summary", "<div>No Alarm</div>"

    cap = cv2.VideoCapture(video)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0: fps = 20

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    temp_out = tempfile.NamedTemporaryFile(delete=False, suffix=".mp4").name
    writer = cv2.VideoWriter(
        temp_out,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width,height)
    )

    seq = deque(maxlen=SEQ_LEN)
    probs = []
    accident_events = 0
    accident_triggered = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(rgb)
        tensor = transform(img)
        seq.append(tensor)

        label = "Collecting..."
        p = 0

        if len(seq) == SEQ_LEN:
            x = torch.stack(list(seq)).unsqueeze(0).to(device)

            with torch.no_grad():
                out = model(x)
                prob = torch.softmax(out, dim=1)[0]
                p = float(prob[ACCIDENT_IDX])

            probs.append(p)

            if p >= threshold:
                label = f"🚨 ACCIDENT ({p:.2f})"
                color = (0,0,255)
                accident_events += 1
                accident_triggered = True
            else:
                label = f"✅ SAFE ({p:.2f})"
                color = (0,255,0)

            cv2.putText(frame,label,(20,45),
                        cv2.FONT_HERSHEY_SIMPLEX,1,color,3)

        writer.write(frame)

    cap.release()
    writer.release()

    graph = make_graph(probs, threshold)

    prediction = "🚨 Accident Detected!" if accident_triggered else "✅ No Accident"

    summary = f"""
Total Frames      : {total_frames}
Accident Events   : {accident_events}
Model             : EfficientNet-B0 + LSTM
Sequence Length   : {SEQ_LEN}
Device            : {device}
"""

    return temp_out, prediction, graph, summary, alarm_html(accident_triggered)


# UI

with gr.Blocks(css=CSS, theme=gr.themes.Soft()) as demo:

    gr.HTML("<div class='main-title'>🚗 Smart Traffic Accident Detection</div>")
    gr.HTML("<div class='sub-title'>EfficientNet-B0 + LSTM +  Dashboard</div>")

    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(label="📤 Upload CCTV Video")
            threshold = gr.Slider(
                0.5,0.99,value=0.8,step=0.01,
                label="Confidence Threshold"
            )
            run_btn = gr.Button("🚀 Run Detection", variant="primary")

        with gr.Column(scale=1):
            pred = gr.Textbox(label="Prediction")
            video_out = gr.Video(label="🎬 Processed Video")
            graph = gr.Plot(label="📈 Confidence Graph")
            summary = gr.Textbox(label="📋 Summary", lines=8)
            alarm_box = gr.HTML(label="🚨 Alarm")

    run_btn.click(
        detect,
        inputs=[video_in, threshold],
        outputs=[video_out, pred, graph, summary, alarm_box]
    )

demo.launch(debug=True, share=True)